# 15 — Capstone: End-to-End RLHF Pipeline

This notebook integrates the full Series 5 curriculum: reward model training → PPO fine-tuning → policy explanation → audit.

In [ ]:
# Full RLHF pipeline: reward model + PPO + attribution + audit report

import torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── 1. Reward Model ─────────────────────────────────────────────────────
print("PHASE 1: Reward Model")
print("="*50)

class RewardModel(nn.Module):
    def __init__(self, dim=256, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim,hidden),nn.ReLU(),nn.Linear(hidden,hidden),nn.ReLU(),nn.Linear(hidden,1))
    def forward(self, x): return self.net(x).squeeze(-1)

w_true = torch.randn(256); w_true /= w_true.norm()
def make_batch(n=64):
    chosen = torch.randn(n, 256); rejected = torch.randn(n, 256)
    sc = w_true @ chosen.T; sr = w_true @ rejected.T
    mask = sc > sr
    c = torch.where(mask.unsqueeze(1), chosen, rejected)
    r = torch.where(mask.unsqueeze(1), rejected, chosen)
    return c, r

rm = RewardModel(dim=256); rm_opt = optim.Adam(rm.parameters(), lr=1e-3)
rm_accs = []
for epoch in range(30):
    chosen, rejected = make_batch(256)
    rc, rr = rm(chosen), rm(rejected)
    loss = -torch.log(torch.sigmoid(rc-rr)+1e-8).mean()
    rm_opt.zero_grad(); loss.backward(); rm_opt.step()
    rm_accs.append((rc>rr).float().mean().item())
    if (epoch+1) % 10 == 0:
        print(f"  Epoch {epoch+1} | Accuracy: {rm_accs[-1]:.3f}")

# ─── 2. PPO Fine-Tuning ──────────────────────────────────────────────────
print("\nPHASE 2: PPO Fine-Tuning")
print("="*50)

class Policy(nn.Module):
    def __init__(self, obs_dim=8, embed_dim=256, hidden=128):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(obs_dim,hidden),nn.Tanh(),nn.Linear(hidden,hidden),nn.Tanh())
        self.mean = nn.Linear(hidden, embed_dim)
        self.log_std = nn.Parameter(torch.zeros(embed_dim)-1)
    def forward(self, obs):
        h = self.enc(obs)
        return self.mean(h), self.log_std.exp().expand(obs.shape[0],-1)
    def sample(self, obs):
        mean, std = self(obs)
        eps = torch.randn_like(mean); z = mean + std*eps
        lp = torch.distributions.Normal(mean, std).log_prob(z).sum(-1)
        return z, lp

sft = Policy(); pol = Policy()
pol.load_state_dict(sft.state_dict())
for p in sft.parameters(): p.requires_grad_(False)
pol_opt = optim.Adam(pol.parameters(), lr=1e-4)
BETA = 0.05

rewards_history, kl_history = [], []
for step in range(200):
    obs = torch.randn(32, 8)
    with torch.no_grad():
        sft_mean, sft_std = sft(obs)
    resp, new_lp = pol.sample(obs)
    reward = rm(resp)
    sft_lp = torch.distributions.Normal(sft_mean, sft_std).log_prob(resp.detach()).sum(-1)
    kl = (new_lp - sft_lp.detach()).mean()
    loss = -(reward - BETA*kl).mean()
    pol_opt.zero_grad(); loss.backward(); pol_opt.step()
    rewards_history.append(reward.mean().item())
    kl_history.append(kl.item())
    if (step+1) % 50 == 0:
        print(f"  Step {step+1} | Reward: {np.mean(rewards_history[-50:]):+.4f} | KL: {np.mean(kl_history[-50:]):.4f}")

# ─── 3. Attribution ──────────────────────────────────────────────────────
print("\nPHASE 3: Policy Attribution")
print("="*50)
obs_s = torch.randn(1, 8, requires_grad=True)
resp, _ = pol.sample(obs_s)
rm(resp).backward()
attrs = obs_s.grad.abs().squeeze().detach().numpy()
top_feat = attrs.argmax()
print(f"  Top obs feature: dim {top_feat} (attr={attrs[top_feat]:.3f})")

# ─── 4. Audit ────────────────────────────────────────────────────────────
print("\nPHASE 4: Audit Report")
print("="*50)
final_reward = np.mean(rewards_history[-50:])
initial_reward = np.mean(rewards_history[:50])
final_kl = np.mean(kl_history[-50:])
print(f"  RM accuracy:          {rm_accs[-1]:.3f}")
print(f"  Reward:               {initial_reward:+.4f} -> {final_reward:+.4f} ({final_reward-initial_reward:+.4f})")
print(f"  Final KL:             {final_kl:.4f} ({'OK' if final_kl < 1.0 else 'EXCEEDED'})")
print(f"  Top attribution feat: dim {top_feat}")


In [ ]:
# Visualisation
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].plot(rm_accs, color='steelblue', marker='o')
axes[0,0].axhline(0.8, color='gray', linestyle='--', alpha=0.5, label='Noisy label ceiling')
axes[0,0].set_title('Phase 1: Reward Model Accuracy', fontweight='bold')
axes[0,0].set_xlabel('Epoch'); axes[0,0].legend()

w = 20
axes[0,1].plot(rewards_history, alpha=0.3, color='steelblue')
if len(rewards_history)>=w: axes[0,1].plot(np.convolve(rewards_history, np.ones(w)/w,'valid'), color='steelblue')
axes[0,1].set_title('Phase 2: PPO Reward', fontweight='bold'); axes[0,1].set_xlabel('Step')

axes[1,0].plot(kl_history, alpha=0.3, color='darkorange')
if len(kl_history)>=w: axes[1,0].plot(np.convolve(kl_history, np.ones(w)/w,'valid'), color='darkorange')
axes[1,0].axhline(1.0, color='red', linestyle='--', alpha=0.5, label='KL limit')
axes[1,0].set_title('Phase 2: KL from SFT', fontweight='bold'); axes[1,0].set_xlabel('Step'); axes[1,0].legend()

obs_names = [f'obs_{i}' for i in range(8)]
axes[1,1].bar(obs_names, attrs, color=['tomato' if i==top_feat else 'steelblue' for i in range(8)])
axes[1,1].set_title('Phase 3: Attribution', fontweight='bold'); axes[1,1].tick_params(axis='x', rotation=45)

plt.suptitle('End-to-End RLHF Pipeline', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.savefig('/tmp/rlhf_capstone.png', dpi=100, bbox_inches='tight'); plt.show()


## Series 5 summary

| NB | Topic | Key concept |
|---|---|---|
| 01 | MDP Fundamentals | GridWorld, policy iteration, Bellman equation |
| 02 | Classical Methods | Tabular Q-learning, TD error |
| 03 | DQN | Experience replay, target network |
| 04 | DQN Variants | Double DQN, Dueling DQN |
| 05 | PPO | Clipped surrogate, GAE, actor-critic |
| 06 | SAC | Max-entropy RL, auto-temperature |
| 07 | Reward Shaping | Potential-based shaping, `reward-hacker` |
| 08 | Safe RL | Constrained MDP, Lagrangian PPO |
| 09 | Multi-Agent RL | Self-play, non-stationarity |
| 10 | Env Validation | `gymcheck` prototype |
| 11 | Custom Envs | Inventory management, Gymnasium interface |
| 12 | RLHF: Reward Model | Bradley-Terry loss, preference data |
| 13 | RLHF: PPO | KL penalty, reference model |
| 14 | Policy Explanation | `policy-lens`: saliency, IG, concept probes |
| 15 | Capstone | RM → PPO → attribution → audit |

Next series: **Series 6 — Bayesian Inference & Modeling**.